# 09 - Customer Segmentation

## Customer Intelligence Platform

This notebook segments customers using K-Means clustering,
creating business-meaningful groups for targeted strategies.

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.load_data import load_features
from src.config import TARGET
from src.segment import (
    prepare_segmentation_features,
    find_optimal_clusters,
    plot_elbow_silhouette,
    fit_kmeans,
    assign_segment_names,
    segment_profiles,
    plot_segment_radar,
)

%matplotlib inline


In [ ]:
df = load_features()


## 1. Prepare Segmentation Features


In [ ]:
X_scaled, scaler, feature_names = prepare_segmentation_features(df)
print(f"Segmentation features: {feature_names}")
print(f"Scaled shape: {X_scaled.shape}")


## 2. Optimal Cluster Selection


In [ ]:
k_range = range(2, 11)
inertias, sil_scores, optimal_k = find_optimal_clusters(X_scaled, k_range)
print(f"Optimal number of clusters: {optimal_k}")

fig = plot_elbow_silhouette(k_range, inertias, sil_scores, optimal_k)
plt.show()


## 3. Fit K-Means


In [ ]:
km, labels = fit_kmeans(X_scaled, n_clusters=5)
df["cluster"] = labels


## 4. Assign Segment Names


In [ ]:
df = assign_segment_names(df, labels, feature_names)
print(f"\nSegment Distribution:")
print(df["segment"].value_counts())


## 5. Segment Profiles


In [ ]:
profiles = segment_profiles(df)
profiles


## 6. Segment Visualization


In [ ]:
fig = plot_segment_radar(df)
plt.show()


In [ ]:
# Churn rate by segment
seg_churn = df.groupby("segment")[TARGET].agg(["mean", "count"]).round(4)
seg_churn.columns = ["Churn Rate", "Customer Count"]
seg_churn = seg_churn.sort_values("Churn Rate", ascending=False)
seg_churn


## Summary

### Final Customer Segments

| Segment | Description | Strategy |
|---------|-------------|----------|
| VIP Loyal | High value, long tenure, low churn risk | Loyalty rewards, upsell |
| VIP At Risk | High value but elevated churn risk | Premium retention package |
| New Customers | Recently joined, uncertain loyalty | Strong onboarding program |
| Low Value | Low spend, low engagement | Standard monitoring |
| Growth Potential | Moderate value, growth opportunity | Service expansion offers |
